# 🌍 AgroCure — Soil Type Classification Model
**MobileNetV2 Transfer Learning | 4 Indian Soil Types → Crop Recommendations**

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. Final cell saves `soil_model.h5` to Google Drive

**Soil Classes:**
- 🔴 Red Soil → Groundnut, Chilli, Millet, Ragi
- ⚫ Black Soil → Cotton, Soybean, Maize, Sunflower
- 🌾 Alluvial Soil → Rice, Wheat, Sugarcane, Banana
- 🟤 Clay Soil → Rice, Jute, Taro

In [ ]:
# ── CELL 1: Imports ────────────────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import json
import shutil
import random

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── CELL 2: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/AgroCure', exist_ok=True)
print('Drive mounted. Model will save to /content/drive/MyDrive/AgroCure/')

In [ ]:
# ── CELL 3: Download Soil Dataset from Kaggle ──────────────────────────────
# Dataset: Soil Types Dataset (4 classes: Alluvial, Black, Clay, Red)
# Kaggle URL: https://www.kaggle.com/datasets/prasanshasatpathy/soil-types

from google.colab import files
print('Upload your kaggle.json file (kaggle.com → Account → API Token):')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

print('Downloading soil dataset...')
# Primary dataset
result = os.system('kaggle datasets download -d prasanshasatpathy/soil-types')

if result != 0:
    print('Trying alternate dataset...')
    os.system('kaggle datasets download -d amandam1/soilclassification')

print('Download complete!')
!ls *.zip

In [ ]:
# ── CELL 4: Extract and inspect ────────────────────────────────────────────
import zipfile

# Extract whichever zip was downloaded
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]
print(f'Found zip files: {zip_files}')

for zf in zip_files:
    print(f'Extracting {zf}...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall('/content/soil_raw')

# Inspect structure
print('\nExtracted structure:')
for root, dirs, files_list in os.walk('/content/soil_raw'):
    level = root.replace('/content/soil_raw', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files_list)} files)')
    if level >= 2:
        break

In [ ]:
# ── CELL 5: Auto-detect soil class folders ────────────────────────────────
# Target: Alluvial, Black, Clay, Red soil (Indian agriculture focused)
TARGET_SOIL_CLASSES = {
    'Alluvial Soil': ['alluvial', 'Alluvial Soil', 'alluvial soil'],
    'Black Soil': ['black', 'Black Soil', 'black soil', 'clay', 'Clay'],  # some datasets use clay for black
    'Clay Soil': ['clay soil', 'Clay Soil', 'loamy', 'Loamy'],
    'Red Soil': ['red', 'Red Soil', 'red soil', 'laterite', 'Laterite'],
}

# Find source directory with soil images
SRC = None
for root, dirs, files_list in os.walk('/content/soil_raw'):
    if len(dirs) >= 2 and any(
        any(alias.lower() in d.lower() for alias in ['alluvial', 'black', 'red', 'clay', 'soil'])
        for d in dirs
    ):
        SRC = root
        print(f'Found soil folders at: {root}')
        print(f'Folders: {dirs}')
        break

if SRC is None:
    print('Could not auto-detect. Listing all folders:')
    for root, dirs, _ in os.walk('/content/soil_raw'):
        if dirs:
            print(f'{root}: {dirs}')

In [ ]:
# ── CELL 6: Build normalized dataset ──────────────────────────────────────
# This cell maps whatever folder names exist → our 4 standard class names

DATASET_DIR = '/content/soil_dataset'
TRAIN_DIR = f'{DATASET_DIR}/train'
VAL_DIR = f'{DATASET_DIR}/val'
SPLIT = 0.8

# Normalized class names (these become CLASS_LABELS in your backend)
FINAL_CLASSES = ['Alluvial Soil', 'Black Soil', 'Clay Soil', 'Red Soil']

# Manual mapping: edit this if auto-detect got it wrong
# Key = folder name in dataset, Value = our final class name
available_folders = os.listdir(SRC) if SRC else []
print(f'Available folders: {available_folders}')

# Auto-build mapping
FOLDER_MAP = {}
for folder in available_folders:
    fl = folder.lower()
    if 'alluvial' in fl:
        FOLDER_MAP[folder] = 'Alluvial Soil'
    elif 'black' in fl or 'dark' in fl:
        FOLDER_MAP[folder] = 'Black Soil'
    elif 'clay' in fl and 'black' not in fl:
        FOLDER_MAP[folder] = 'Clay Soil'
    elif 'red' in fl or 'laterite' in fl:
        FOLDER_MAP[folder] = 'Red Soil'
    elif 'loam' in fl or 'sandy' in fl:
        FOLDER_MAP[folder] = 'Clay Soil'  # fallback

print(f'\nFolder mapping: {FOLDER_MAP}')
print('\n⚠️  If mapping looks wrong, edit FOLDER_MAP manually above and re-run!')

In [ ]:
# ── CELL 7: If dataset missing classes, generate synthetic fallback ────────
# Some soil datasets only have 3-4 classes. This handles that gracefully.

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

total_train, total_val = 0, 0
built_classes = set()

for folder, class_name in FOLDER_MAP.items():
    src_folder = os.path.join(SRC, folder)
    if not os.path.isdir(src_folder):
        continue

    images = [f for f in os.listdir(src_folder)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG'))]

    if not images:
        continue

    random.shuffle(images)
    split_idx = int(len(images) * SPLIT)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    os.makedirs(f'{TRAIN_DIR}/{class_name}', exist_ok=True)
    os.makedirs(f'{VAL_DIR}/{class_name}', exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(src_folder, img), f'{TRAIN_DIR}/{class_name}/{img}')
    for img in val_imgs:
        shutil.copy(os.path.join(src_folder, img), f'{VAL_DIR}/{class_name}/{img}')

    total_train += len(train_imgs)
    total_val += len(val_imgs)
    built_classes.add(class_name)
    print(f'{class_name}: {len(train_imgs)} train | {len(val_imgs)} val')

FINAL_CLASSES = sorted(list(built_classes))
NUM_CLASSES = len(FINAL_CLASSES)
print(f'\nFinal classes ({NUM_CLASSES}): {FINAL_CLASSES}')
print(f'Total: {total_train} train | {total_val} val')

In [ ]:
# ── CELL 8: Data generators ────────────────────────────────────────────────
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
NUM_CLASSES = len(class_indices)

print('\nClass index mapping (USE THIS in soil_predict.py):')
for i in range(NUM_CLASSES):
    print(f'  {i}: {idx_to_class[i]}')

In [ ]:
# ── CELL 9: Visualize sample training images ───────────────────────────────
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

idx = 0
for class_name in list(idx_to_class.values())[:4]:
    cls_dir = f'{TRAIN_DIR}/{class_name}'
    imgs = os.listdir(cls_dir)[:2]
    for img_name in imgs:
        img = PILImage.open(f'{cls_dir}/{img_name}').resize((224, 224))
        axes[idx].imshow(img)
        axes[idx].set_title(class_name, fontsize=9)
        axes[idx].axis('off')
        idx += 1

plt.suptitle('Sample Soil Images (Training Set)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/soil_samples.png', dpi=120)
plt.show()
print('Sample images displayed!')

In [ ]:
# ── CELL 10: Build model ───────────────────────────────────────────────────
def build_soil_model(num_classes):
    base = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output)
    return model, base

model, base_model = build_soil_model(NUM_CLASSES)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Model built. Params: {model.count_params():,}')

In [ ]:
# ── CELL 11: Phase 1 — Train classifier head ───────────────────────────────
callbacks = [
    ModelCheckpoint(
        '/content/best_soil_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('Phase 1: Training head only...')
history1 = model.fit(
    train_gen,
    epochs=12,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f'Best val accuracy: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# ── CELL 12: Phase 2 — Fine-tune ──────────────────────────────────────────
base_model.trainable = True
fine_tune_from = len(base_model.layers) - 40

for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 2: Fine-tuning...')
history2 = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f'Best fine-tune val accuracy: {max(history2.history["val_accuracy"]):.4f}')

In [ ]:
# ── CELL 13: Evaluate ─────────────────────────────────────────────────────
from sklearn.metrics import classification_report

best_model = tf.keras.models.load_model('/content/best_soil_model.h5')

val_gen.reset()
loss, acc = best_model.evaluate(val_gen, verbose=0)
print(f'Validation Accuracy: {acc*100:.2f}%')

val_gen.reset()
preds = best_model.predict(val_gen, verbose=1)
pred_classes = np.argmax(preds, axis=1)
true_classes = val_gen.classes
class_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print('\nClassification Report:')
print(classification_report(true_classes, pred_classes, target_names=class_names))

In [ ]:
# ── CELL 14: Plot training curves ─────────────────────────────────────────
all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']

plt.figure(figsize=(10, 4))
plt.plot(all_acc, label='Train Accuracy')
plt.plot(all_val_acc, label='Val Accuracy')
plt.axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Soil Classification Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AgroCure/training_curves_soil.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 15: Save model + crop recommendation map ─────────────────────────

# Soil → Crop recommendation mapping (used by your backend)
SOIL_CROP_MAP = {
    'Alluvial Soil': {
        'description': 'Fertile, found in river plains. Best for high-yield crops.',
        'ph_range': '6.5 - 8.0',
        'recommended_crops': ['Rice', 'Wheat', 'Sugarcane', 'Banana', 'Mango', 'Vegetables'],
        'avoid_crops': ['Groundnut', 'Millet'],
        'fertilizer_tip': 'Moderate NPK. Add organic matter for long-term health.',
        'irrigation': 'Medium water requirement',
        'season': ['kharif', 'rabi']
    },
    'Black Soil': {
        'description': 'Rich in clay, retains moisture. Ideal for cotton belt of India.',
        'ph_range': '7.0 - 8.5',
        'recommended_crops': ['Cotton', 'Soybean', 'Maize', 'Sunflower', 'Jowar', 'Wheat'],
        'avoid_crops': ['Rice', 'Groundnut'],
        'fertilizer_tip': 'Low nitrogen needed. High in calcium and magnesium naturally.',
        'irrigation': 'Low — retains moisture well',
        'season': ['kharif']
    },
    'Clay Soil': {
        'description': 'Heavy, water-retentive soil. Good for flooded crops.',
        'ph_range': '5.5 - 7.0',
        'recommended_crops': ['Rice', 'Jute', 'Taro', 'Lotus', 'Water Chestnut'],
        'avoid_crops': ['Groundnut', 'Carrot', 'Radish'],
        'fertilizer_tip': 'Add sand/grit to improve drainage. Use organic compost.',
        'irrigation': 'Low — retains too much water, improve drainage',
        'season': ['kharif']
    },
    'Red Soil': {
        'description': 'Iron-rich, well-drained. Common in Tamil Nadu, Karnataka, Andhra.',
        'ph_range': '5.5 - 7.5',
        'recommended_crops': ['Groundnut', 'Chilli', 'Millet', 'Ragi', 'Tomato', 'Tobacco'],
        'avoid_crops': ['Rice', 'Wheat'],
        'fertilizer_tip': 'Deficient in nitrogen and humus. Apply FYM + NPK regularly.',
        'irrigation': 'Medium — poor water retention, needs regular irrigation',
        'season': ['kharif', 'rabi']
    }
}

# Save model
MODEL_SAVE_PATH = '/content/drive/MyDrive/AgroCure/soil_model.h5'
best_model.save(MODEL_SAVE_PATH)
print(f'Soil model saved to: {MODEL_SAVE_PATH}')

# Save class labels
labels_path = '/content/drive/MyDrive/AgroCure/soil_class_labels.json'
with open(labels_path, 'w') as f:
    json.dump(idx_to_class, f, indent=2)

# Save crop map
crop_map_path = '/content/drive/MyDrive/AgroCure/soil_crop_map.json'
with open(crop_map_path, 'w') as f:
    json.dump(SOIL_CROP_MAP, f, indent=2)

print('\n=== COPY THIS INTO soil_predict.py SOIL_LABELS ===')
print('SOIL_LABELS = [')
for i in range(NUM_CLASSES):
    print(f'    "{idx_to_class[i]}",  # index {i}')
print(']')

print(f'\nFinal accuracy: {acc*100:.2f}%')
print('Done! Download soil_model.h5 from Google Drive/AgroCure/')

In [ ]:
# ── CELL 16: Add soil prediction endpoint helper ───────────────────────────
# This is the code you'll paste into your FastAPI backend

BACKEND_CODE = '''
# ── model/soil_predict.py (ADD THIS FILE TO YOUR BACKEND) ─────────────────
import numpy as np
import tensorflow as tf
from pathlib import Path

SOIL_MODEL_PATH = Path(__file__).parent / "soil_model.h5"

# UPDATE THESE after training — copy from Cell 15 output above
SOIL_LABELS = [
    "Alluvial Soil",
    "Black Soil",
    "Clay Soil",
    "Red Soil",
]

SOIL_CROP_MAP = {
    "Alluvial Soil": {"crops": ["Rice", "Wheat", "Sugarcane", "Banana"], "irrigation": "medium"},
    "Black Soil":    {"crops": ["Cotton", "Soybean", "Maize", "Sunflower"], "irrigation": "low"},
    "Clay Soil":     {"crops": ["Rice", "Jute", "Taro"], "irrigation": "low"},
    "Red Soil":      {"crops": ["Groundnut", "Chilli", "Millet", "Ragi"], "irrigation": "medium"},
}

_soil_model = None

def load_soil_model():
    global _soil_model
    if SOIL_MODEL_PATH.exists():
        _soil_model = tf.keras.models.load_model(str(SOIL_MODEL_PATH))
        print("[soil] Model loaded")
    else:
        print("[soil] Model not found — demo mode")

def predict_soil(image_array: np.ndarray) -> dict:
    if _soil_model is None:
        return {"soil_type": "Red Soil", "confidence": 0.91, "demo_mode": True,
                "recommended_crops": SOIL_CROP_MAP["Red Soil"]["crops"]}
    probs = _soil_model.predict(image_array, verbose=0)[0]
    top_idx = int(np.argmax(probs))
    soil_type = SOIL_LABELS[top_idx]
    return {
        "soil_type": soil_type,
        "confidence": round(float(probs[top_idx]), 4),
        "recommended_crops": SOIL_CROP_MAP[soil_type]["crops"],
        "irrigation_need": SOIL_CROP_MAP[soil_type]["irrigation"],
        "demo_mode": False,
    }
'''

print(BACKEND_CODE)
print('\nAlso add this route to api/app.py:')
print('''
@app.post("/api/scan/soil")
async def scan_soil(image: UploadFile = File(...)):
    image_bytes = await image.read()
    is_valid, err = validate_image(image_bytes)
    if not is_valid:
        raise HTTPException(status_code=400, detail=err)
    img_array = preprocess_image(image_bytes)
    result = predict_soil(img_array)
    return result
''')